# Z-Scores Calculation

Notebook to compute Z-scores from model predictions to evaluate term importance.

In [52]:
import numpy as np
import pandas as pd
from scipy import stats
from tqdm import tqdm
import os
import json
import shutup; shutup.please()
dataset = "Analgesics-induced_acute_liver_failure"
dirpath = os.path.join(r'/home/hsdslab/Documents/Csabi/Pharma_crossval/DeepCausalPV-master-main/dat',dataset,'proc')

# Functions

In [53]:
def calculate_z_scores(df, terms_dict, score_col):
    """
    Calculate Z-scores for unique terms in specified columns of a DataFrame.
    
    Args:
    df (pd.DataFrame): The input DataFrame containing the data.
    terms_dict (dict): A dictionary where keys are column names and values are lists of unique terms to analyze.
    
    Returns:
    pd.DataFrame: A DataFrame containing columns, terms, and their corresponding Z-scores, sorted by Z-score.
    """
    results = []

    for column_name, terms in terms_dict.items():
        # Initialize a dictionary to store the Z-scores for this column
        z_scores = {}
        
        # Filter out rows where the specified column is NaN
        filtered_df = df[df[column_name].notna()]

        # Iterate through each term in the list
        for term in terms:
            # Create a mask for the rows containing the term
            mask_present = filtered_df[column_name].str.contains(term, na=False, regex=False)
            
            # Get probabilities for both parts
            prob_present = filtered_df.loc[mask_present, score_col]
            prob_absent = filtered_df.loc[~mask_present, score_col]
            
            # Check if we have enough data to perform the test
            if len(prob_present) >= 30 and len(prob_absent) >= 30:
                # Calculate means and standard deviations
                mean_present = prob_present.mean()
                mean_absent = prob_absent.mean()
                std_present = prob_present.std(ddof=1)  # Sample standard deviation
                std_absent = prob_absent.std(ddof=1)    # Sample standard deviation
                
                # Calculate the Z-score
                z_score = (mean_present - mean_absent) / np.sqrt((std_present**2 / len(prob_present)) + (std_absent**2 / len(prob_absent)))
                
                # Store the result
                z_scores[term] = z_score

        # Order Z-scores dictionary by values
        z_scores = dict(sorted(z_scores.items(), key=lambda item: item[1], reverse=True))

        # Append results for this column to the results list
        for term, z_score in z_scores.items():
            results.append({'Column': column_name, 'Term': term, 'Z-score': z_score})

    # Convert the results list into a DataFrame
    z_scores_df = pd.DataFrame(results)

    # Sort the DataFrame by Z-score in descending order
    try:
        z_scores_df = z_scores_df.sort_values(by='Z-score', ascending=False)
        # drop rows where Z_score is less then 1.64 or nan
        z_scores_df = z_scores_df[(z_scores_df['Z-score'] > 1.64) | (z_scores_df['Z-score'].isna())]
    except:
        z_scores_df = None
        
    if z_scores_df is not None:
        zscores_list = (z_scores_df["Column"] + "_" + z_scores_df["Term"]).tolist()
    else:
        zscores_list = []
    return zscores_list

In [54]:
def calculate_ppr(df, terms_dict):
    results = []
    for column_name, terms in terms_dict.items():
        pprs = {}
        filtered_df = df[df[column_name].notna()]
        
        for term in terms:
            # Create a mask for the rows containing the term
            mask_present = filtered_df[column_name].str.contains(term, na=False, regex=False)
            
            labels_present = filtered_df.loc[mask_present, "label"]
            labels_absent = filtered_df.loc[~mask_present, "label"]
            
            if len(labels_present) >= 30 and len(labels_absent) >= 30:
            
                A = labels_present.sum()
                B = len(labels_present) - A

                C = labels_absent.sum()
                D = len(labels_absent) - C

                if A>3:
                    # Calculate the PPR
                    ppr = (A * (C + D)) / ((A + B) * C)
                    
                    EA = (A + B) * (A + C) / (A + B + C + D)
                    EB = (A + B) * (B + D) / (A + B + C + D)
                    EC = (C + D) * (A + C) / (A + B + C + D)
                    ED = (C + D) * (B + D) / (A + B + C + D)
                    
                    # calculate chi-square statistic
                    chi2 = (A - EA)**2 / EA + (B - EB)**2 / EB + (C - EC)**2 / EC + (D - ED)**2 / ED
                    
                    pprs[term] = (ppr, chi2)
        
        for term, ppr in pprs.items():
            results.append({'Column': column_name, 'Term': term, 'ppr': ppr[0], 'chi2': ppr[1]})
            
        pprs_df = pd.DataFrame(results)
        
    try:
        pprs_df = pprs_df.sort_values(by='ppr', ascending=False)
        pprs_df = pprs_df[pprs_df['ppr'] >= 2]
        pprs_df = pprs_df[pprs_df['chi2'] >= 4]
    except:
        pprs_df = None
    
    if pprs_df is not None:
        pprs_list = (pprs_df["Column"] + "_" + pprs_df["Term"]).tolist()
    else:
        pprs_list = []
    
    return pprs_list

In [55]:
def calculate_ror(df, terms_dict):
    results = []
    for column_name, terms in terms_dict.items():
        rors = {}
        filtered_df = df[df[column_name].notna()]
        
        for term in terms:
            # Create a mask for the rows containing the term
            mask_present = filtered_df[column_name].str.contains(term, na=False, regex=False)
            
            labels_present = filtered_df.loc[mask_present, "label"]
            labels_absent = filtered_df.loc[~mask_present, "label"]
            
            if len(labels_present) >= 30 and len(labels_absent) >= 30 and labels_present.sum() > 3:
                A = labels_present.sum()
                B = len(labels_present) - A
                
                C = labels_absent.sum()
                D = len(labels_absent) - B
                
                ror = (A*D) / (B*C)
                conf = np.exp([np.log(ror) - 1.96 * np.sqrt(1/A + 1/B + 1/C + 1/D), np.log(ror) + 1.96 * np.sqrt(1/A + 1/B + 1/C + 1/D)])
                
                rors[term] = conf
                
        
        for term, conf in rors.items():
            results.append({'Column': column_name, 'Term': term, 'conf_l': conf[0], 'conf_u': conf[1]})
            
        rors_df = pd.DataFrame(results)
        
    try:
        rors_df = rors_df.sort_values(by='conf_l', ascending=False)
        rors_df = rors_df[rors_df['conf_l'] > 1]
    except:
        rors_df = None
        
    if rors_df is not None:
        rors_list = (rors_df["Column"] + "_" + rors_df["Term"]).tolist()
    else:
        rors_list = []
            
    return rors_list

In [56]:
def calculate_ebgm(df, terms_dict):
    results = []
    for column_name, terms in terms_dict.items():
        ebgms = {}
        filtered_df = df[df[column_name].notna()]
        
        for term in terms:
            # Create a mask for the rows containing the term
            mask_present = filtered_df[column_name].str.contains(term, na=False, regex=False)
            
            labels_present = filtered_df.loc[mask_present, "label"]
            labels_absent = filtered_df.loc[~mask_present, "label"]
            
            if len(labels_present) >= 30 and len(labels_absent) >= 30 and labels_present.sum() > 3:
                A = labels_present.sum()
                B = len(labels_present) - A
                
                C = labels_absent.sum()
                D = len(labels_absent) - B
                
                ebgm = (A*(A+B+C+D)) / ((A+B)*(A+C))
                conf = np.exp(np.log(ebgm) - 1.64 * np.sqrt(1/A + 1/B + 1/C + 1/D))
                
                ebgms[term] = conf
                
        
        for term, conf in ebgms.items():
            results.append({'Column': column_name, 'Term': term, 'conf_l': conf})
            
        ebgm_df = pd.DataFrame(results)
        
    try:
        ebgm_df = ebgm_df.sort_values(by='conf_l', ascending=False)
        ebgm_df = ebgm_df[ebgm_df['conf_l'] > 2]
    except:
        ebgm_df = None
    
    if ebgm_df is not None:
        ebgm_list = (ebgm_df["Column"] + "_" + ebgm_df["Term"]).tolist()
    else:
        ebgm_list = []
    
    return ebgm_list

In [57]:
def process_idx(crossval_idx):
    xgb = pd.read_csv(os.path.join(dirpath,"cross_val_xgb", f"df_res{crossval_idx[0]}{crossval_idx[1]}.csv"), index_col=0)
    xgb.drop("label", axis=1, inplace=True)
    biobert_temp = pd.read_csv(os.path.join(dirpath,"cross_val_biobert_temp", f"df_res{crossval_idx[0]}{crossval_idx[1]}.csv"), index_col=0)
    biobert_temp.drop("label", axis=1, inplace=True)
    biobert_llm = pd.read_csv(os.path.join(dirpath,"cross_val_biobert_llm", f"df_res{crossval_idx[0]}{crossval_idx[1]}.csv"), index_col=0)
    biobert_llm.drop("label", axis=1, inplace=True)
    albert_temp = pd.read_csv(os.path.join(dirpath,"cross_val_albert_temp", f"df_res{crossval_idx[0]}{crossval_idx[1]}.csv"), index_col=0)
    albert_temp.drop("label", axis=1, inplace=True)
    df = pd.read_csv(os.path.join(dirpath,'df_together.csv'), index_col=0)
    df.drop("Temp_sentence", axis=1, inplace=True)
    df = df.loc[xgb.index]
    df = pd.concat([df, xgb, biobert_temp, biobert_llm, albert_temp], axis=1)
    
    age_terms = list(set(list(df["age"].dropna())))
    dose_terms = list(set(list(df["dose"].dropna())))
    gender_terms = ["Male","Female"]

    # Function to get sorted unique terms from a specified column
    def get_sorted_unique_terms(column):
        terms = df[column].dropna().str.split(',').explode()  # Split by commas and explode
        terms = terms.str.strip()                     # Strip leading/trailing whitespace
        unique_terms = terms.unique().tolist()        # Get unique terms as a list
        return sorted(unique_terms)                   # Sort the terms alphabetically

    # Get sorted unique terms from each specified column
    sorted_unique_terms_psd = get_sorted_unique_terms('psd')
    sorted_unique_terms_ssd = get_sorted_unique_terms('ssd')
    sorted_unique_terms_ccd = get_sorted_unique_terms('ccd')
    sorted_unique_terms_idrug = get_sorted_unique_terms('idrug')
    sorted_unique_terms_indication = get_sorted_unique_terms('indication')
    sorted_unique_terms_outcome = get_sorted_unique_terms('outcome')
    sorted_unique_age_terms = sorted(age_terms)
    sorted_unique_dose_terms = sorted(dose_terms)
    sorted_unique_gender_terms = sorted(gender_terms)
    
    terms_dict = {
        'psd': sorted_unique_terms_psd,
        'ssd': sorted_unique_terms_ssd,
        'ccd': sorted_unique_terms_ccd,
        'idrug': sorted_unique_terms_idrug,
        'indication': sorted_unique_terms_indication,
        'outcome': sorted_unique_terms_outcome,
        'age': sorted_unique_age_terms,
        'dose': sorted_unique_dose_terms,
        'gender': sorted_unique_gender_terms
    }
    
    results = {}
    
    model_names = ['xgb', 'xgb_cal', 'biobert_temp', 'biobert_temp_cal', 'biobert_llm',
       'biobert_llm_cal', 'albert_temp', 'albert_temp_cal']
    
    for model_name in model_names:
        z_scores_list = calculate_z_scores(df, terms_dict, model_name)
        results[model_name] = z_scores_list
        
    results["prr"] = calculate_ppr(df, terms_dict)
    results["ror"] = calculate_ror(df, terms_dict)
    results["ebgm"] = calculate_ebgm(df, terms_dict)
        
    return results

# Processing

In [58]:
all_results = {}

for i in range(5):
    for j in tqdm(range(5)):
        if i!=j:
            all_results[f"{i}{j}"] = process_idx((i,j))
            
with open(os.path.join(dirpath, "all_sigterms.json"), "w") as f:
    json.dump(all_results, f)

  0%|          | 0/5 [00:00<?, ?it/s]/tmp/ipykernel_27274/2342587909.py:22: RuntimeWarning: invalid value encountered in log
  conf = np.exp([np.log(ror) - 1.96 * np.sqrt(1/A + 1/B + 1/C + 1/D), np.log(ror) + 1.96 * np.sqrt(1/A + 1/B + 1/C + 1/D)])
 40%|████      | 2/5 [01:06<01:39, 33.23s/it]/tmp/ipykernel_27274/2342587909.py:22: RuntimeWarning: invalid value encountered in log
  conf = np.exp([np.log(ror) - 1.96 * np.sqrt(1/A + 1/B + 1/C + 1/D), np.log(ror) + 1.96 * np.sqrt(1/A + 1/B + 1/C + 1/D)])
 60%|██████    | 3/5 [02:14<01:35, 47.58s/it]/tmp/ipykernel_27274/2342587909.py:22: RuntimeWarning: invalid value encountered in log
  conf = np.exp([np.log(ror) - 1.96 * np.sqrt(1/A + 1/B + 1/C + 1/D), np.log(ror) + 1.96 * np.sqrt(1/A + 1/B + 1/C + 1/D)])
 80%|████████  | 4/5 [03:20<00:54, 54.75s/it]/tmp/ipykernel_27274/2342587909.py:22: RuntimeWarning: invalid value encountered in log
  conf = np.exp([np.log(ror) - 1.96 * np.sqrt(1/A + 1/B + 1/C + 1/D), np.log(ror) + 1.96 * np.sqrt(1/A +

In [60]:
len(list(all_results.keys()))

20